<style>
.jp-RenderedHTMLCommon h1{color:#ff9900;font-size:2.25em}.jp-RenderedHTMLCommon h2{color:#2563a8}
.jp-RenderedHTMLCommon blockquote{border-left:6px solid #ff9900;background:#fff7e8;padding:.6em 1em}
.jp-RenderedHTMLCommon table{font-size:.9em}.jp-RenderedHTMLCommon code{color:#9a3412}
</style>

# Parsing JSON and XML with AWS Glue
## Staged e-commerce orders and invoices

**Copy → save → upload → classify → crawl → validate**

> Goal: turn nested JSON and XML documents into accurate Glue Data Catalog table definitions.


# Outcomes

This walkthrough establishes how to:

- distinguish JSON documents, JSON arrays, and JSON Lines;
- identify the repeating record node in JSON and XML;
- create custom JSON and XML classifiers;
- stage sample orders and invoices in Amazon S3;
- configure separate crawlers and Catalog tables;
- verify nested structures, arrays, attributes, datatypes, and source locations;
- diagnose malformed documents and incorrect record paths.


# Target layout

Use separate prefixes because JSON orders and XML invoices have different formats and record structures.

```text
s3://gksdatalake/bronze/ecommerce/
├── orders_json/
│   └── orders.json
└── invoices_xml/
    └── invoices.xml
```

Suggested resources:

- database: `gks_ecommerce_bronze`
- JSON classifier: `gks_orders_json_classifier`
- XML classifier: `gks_invoices_xml_classifier`
- crawlers: `gks_orders_json_crawler`, `gks_invoices_xml_crawler`


# JSON structures: choose deliberately

| Structure | Shape | Record-path implication |
|---|---|---|
| One object | `{ "orderId": ... }` | root object is one record |
| Array of objects | `[ {...}, {...} ]` | array elements are records |
| Wrapped array | `{ "orders": [ ... ] }` | use a path such as `$.orders[*]` |
| JSON Lines / NDJSON | one JSON object per line | not the same document shape as a wrapped array |

This example uses a **wrapped array** so the classifier must explicitly identify the repeating `orders` elements.


# Stage `orders.json`

Copy the entire block into a plain-text editor. Save it as **`orders.json`** using UTF-8 encoding.

```json
{
  "batchId": "ORD-2026-09-01-A",
  "source": "web-store",
  "orders": [
    {
      "orderId": "O-1001",
      "orderTimestamp": "2026-09-01T09:15:00Z",
      "status": "PAID",
      "customer": {
        "customerId": "C-501",
        "name": "Ananya Rao",
        "email": "ananya@example.com"
      },
      "shippingAddress": {
        "city": "Bengaluru",
        "state": "Karnataka",
        "postalCode": "560001",
        "country": "IN"
      },
      "items": [
        {"sku": "SKU-BOOK-01", "quantity": 2, "unitPrice": 499.50},
        {"sku": "SKU-BAG-02", "quantity": 1, "unitPrice": 899.00}
      ],
      "currency": "INR",
      "orderTotal": 1898.00,
      "couponCode": null
    },
    {
      "orderId": "O-1002",
      "orderTimestamp": "2026-09-01T10:05:30Z",
      "status": "SHIPPED",
      "customer": {
        "customerId": "C-502",
        "name": "Ravi Shah",
        "email": "ravi@example.com"
      },
      "shippingAddress": {
        "city": "Pune",
        "state": "Maharashtra",
        "postalCode": "411001",
        "country": "IN"
      },
      "items": [
        {"sku": "SKU-MUG-03", "quantity": 3, "unitPrice": 249.00}
      ],
      "currency": "INR",
      "orderTotal": 747.00,
      "couponCode": "WELCOME10"
    }
  ]
}
```


# Validate the JSON before upload

Confirm these conditions:

- property names and string values use double quotes;
- there are no trailing commas;
- braces `{}` and brackets `[]` are balanced;
- numeric values are not quoted unless intentionally strings;
- `null` is unquoted;
- the complete file is one valid JSON document;
- encoding is UTF-8 and the file is not saved as `orders.json.txt`.

The batch-level `batchId` and `source` sit outside `orders`. A classifier path focused on `orders[*]` will define rows from each order, not automatically copy wrapper fields into every row.


# JSON record path

The repeating records are the elements inside the `orders` array:

```text
$.orders[*]
```

Interpretation:

- `$` — document root
- `.orders` — child property named `orders`
- `[*]` — every array element becomes a candidate record

Expected nested schema:

```text
orderId: string
customer: struct<customerId:string,name:string,email:string>
shippingAddress: struct<city:string,state:string,postalCode:string,country:string>
items: array<struct<sku:string,quantity:...,unitPrice:...>>
orderTotal: numeric
couponCode: string/null
```


# Create the JSON classifier

In AWS Glue, open **Data Catalog → Classifiers → Add classifier**.

- **Name:** `gks_orders_json_classifier`
- **Type:** JSON
- **JSON path:** `$.orders[*]`

Create the classifier before the crawler. A custom JSON classifier supplies the record-selection path; the crawler uses the selected objects to infer fields and types.

If the file instead contained a root array, the path would commonly select root elements, such as `$[*]`. Match the path to the actual document structure.


# XML structure: identify the row element

XML has one document root, but the table rows come from a repeating element.

```xml
<invoiceBatch>
  <invoices>
    <invoice> ... </invoice>
    <invoice> ... </invoice>
  </invoices>
</invoiceBatch>
```

For this example, the repeating record element is:

```text
invoice
```

The XML classifier's **row tag** identifies that element. It is a tag name, not an XPath expression.


# Stage `invoices.xml`

Copy the entire block into a plain-text editor. Save it as **`invoices.xml`** using UTF-8 encoding.

```xml
<?xml version="1.0" encoding="UTF-8"?>
<invoiceBatch batchId="INV-2026-09-01-A" generatedAt="2026-09-01T12:00:00Z">
  <invoices>
    <invoice invoiceId="INV-9001" status="ISSUED">
      <orderId>O-1001</orderId>
      <invoiceDate>2026-09-01</invoiceDate>
      <customer customerId="C-501">
        <name>Ananya Rao</name>
        <email>ananya@example.com</email>
      </customer>
      <lineItems>
        <lineItem lineNumber="1">
          <sku>SKU-BOOK-01</sku>
          <description>Data Engineering Handbook</description>
          <quantity>2</quantity>
          <unitPrice>499.50</unitPrice>
          <lineTotal>999.00</lineTotal>
        </lineItem>
        <lineItem lineNumber="2">
          <sku>SKU-BAG-02</sku>
          <description>Travel Bag</description>
          <quantity>1</quantity>
          <unitPrice>899.00</unitPrice>
          <lineTotal>899.00</lineTotal>
        </lineItem>
      </lineItems>
      <tax amount="341.64" rate="18.0" />
      <currency>INR</currency>
      <subtotal>1898.00</subtotal>
      <grandTotal>2239.64</grandTotal>
    </invoice>
    <invoice invoiceId="INV-9002" status="ISSUED">
      <orderId>O-1002</orderId>
      <invoiceDate>2026-09-01</invoiceDate>
      <customer customerId="C-502">
        <name>Ravi Shah</name>
        <email>ravi@example.com</email>
      </customer>
      <lineItems>
        <lineItem lineNumber="1">
          <sku>SKU-MUG-03</sku>
          <description>Ceramic Mug</description>
          <quantity>3</quantity>
          <unitPrice>249.00</unitPrice>
          <lineTotal>747.00</lineTotal>
        </lineItem>
      </lineItems>
      <tax amount="134.46" rate="18.0" />
      <currency>INR</currency>
      <subtotal>747.00</subtotal>
      <grandTotal>881.46</grandTotal>
    </invoice>
  </invoices>
</invoiceBatch>
```


# Validate the XML before upload

Confirm:

- exactly one root element: `<invoiceBatch>`;
- every opening tag has a matching closing tag;
- attribute values are quoted;
- reserved characters in values are escaped: `&amp;`, `&lt;`, `&gt;`;
- element names use consistent case;
- XML declaration and file encoding agree;
- the repeating `<invoice>` elements are not self-closing;
- the file is not saved as `invoices.xml.txt`.

XML is case-sensitive: `invoice`, `Invoice`, and `INVOICE` are different tags.


# XML elements and attributes

The staged document intentionally contains both:

```xml
<invoice invoiceId="INV-9001" status="ISSUED">
  <orderId>O-1001</orderId>
</invoice>
```

- `invoiceId` and `status` are **attributes**.
- `orderId` is a child **element**.
- `lineItems` contains repeated `lineItem` children.
- `tax` is a self-closing nested element with attributes.

Inspect the resulting Catalog schema carefully. XML readers can represent attributes and nested content differently from JSON fields, and downstream names may not look exactly like the source markup.


# Create the XML classifier

In **Data Catalog → Classifiers → Add classifier**:

- **Name:** `gks_invoices_xml_classifier`
- **Type:** XML
- **Classification:** `xml` or a clear invoice-specific label if requested
- **Row tag:** `invoice`

Do not enter `/invoiceBatch/invoices/invoice`; the row tag is the element name that represents one record.

The classifier should return one logical row for each repeating `<invoice>...</invoice>` element.


# Upload the staged files

Upload each file to its exact dataset prefix:

```text
orders.json
→ s3://gksdatalake/bronze/ecommerce/orders_json/orders.json

invoices.xml
→ s3://gksdatalake/bronze/ecommerce/invoices_xml/invoices.xml
```

After upload, verify:

- object key and extension;
- nonzero object size;
- expected Region/account;
- encryption settings and crawler-role access;
- no temporary, backup, or unrelated files in either prefix.

Avoid uploading the notebook itself into a crawler target.


# Create the Catalog database

In **Data Catalog → Databases**, create:

```text
gks_ecommerce_bronze
```

Suggested description:

```text
Raw e-commerce order and invoice metadata
```

The database is a regional metadata namespace. It does not store the JSON or XML content and does not create the S3 folders.


# Crawler design: keep formats separate

Create two crawlers rather than one mixed-format crawler:

| Crawler | S3 target | Custom classifier |
|---|---|---|
| `gks_orders_json_crawler` | `.../orders_json/` | `gks_orders_json_classifier` |
| `gks_invoices_xml_crawler` | `.../invoices_xml/` | `gks_invoices_xml_classifier` |

Both can write to `gks_ecommerce_bronze`.

This design isolates failures, produces clearer table names, avoids format-grouping surprises, and allows different classifier or schema-change behavior later.


# Configure each crawler

For each crawler, set the following intent in the Glue Console:

1. Select **S3** as the source.
2. Enter the dataset-specific prefix.
3. Choose or create a crawler IAM role with scoped S3 read access.
4. Attach the matching custom classifier first.
5. Select `gks_ecommerce_bronze` as the output database.
6. Add a table prefix only if naming collisions require one.
7. Use **on demand** scheduling for this staged example.
8. Start with a full crawl and an understood schema-change policy.

Review the final summary before creation.


# IAM access model

The crawler role needs permission to inspect its assigned S3 prefix and write metadata through Glue.

Conceptually verify:

- `s3:ListBucket` for the relevant prefix;
- `s3:GetObject` for staged objects;
- KMS decrypt permission if customer-managed encryption is used;
- Glue Data Catalog permissions required by the crawler;
- a valid Glue trust relationship on the role.

The console operator additionally needs permission to create resources and pass the role to Glue. Network connections are not normally required for standard S3 targets.


# Run the JSON crawler

Start `gks_orders_json_crawler`, wait for **Ready**, then inspect its history.

Open the created table and verify:

- location ends with `/orders_json/`;
- classification reflects JSON;
- each `orders[*]` element is treated as a row;
- `customer` and `shippingAddress` are nested structures;
- `items` is an array of structures;
- `orderTotal` is numeric;
- nullable `couponCode` remains usable;
- wrapper fields are absent unless the selected record structure includes them.

A successful crawl still requires schema inspection.


# Run the XML crawler

Start `gks_invoices_xml_crawler`, wait for **Ready**, then inspect the created table.

Verify:

- location ends with `/invoices_xml/`;
- classification reflects XML;
- one logical record is based on each `<invoice>` element;
- invoice attributes are present in the inferred structure;
- customer and line-item nesting is retained;
- repeated `<lineItem>` elements are represented as a collection;
- decimal-looking values have sensible inferred types;
- outer `invoiceBatch` attributes are not assumed to repeat automatically.


# Parsing is not flattening

After crawling, nested data may remain nested:

```text
orders.customer.name
orders.items[*].sku
invoices.lineItems.lineItem[*].quantity
```

The crawler creates metadata describing structure. It does not automatically:

- explode every array into separate rows;
- join orders to invoices;
- convert epoch or ISO strings to a chosen timestamp type;
- enforce monetary precision;
- standardize attribute and element naming;
- create curated Parquet output.

Those are transformation responsibilities for a later Glue job or query.


# Datatype inference: verify business meaning

| Value | Possible inferred representation | Required decision |
|---|---|---|
| `"2026-09-01T09:15:00Z"` | string or timestamp-like | parse explicitly and choose timezone behavior |
| `499.50` | floating/decimal-like | choose decimal precision for money |
| `"560001"` | string | keep leading-zero-safe postal semantics |
| `null` coupon | nullable/choice behavior | define expected optionality |
| one vs many XML children | struct vs array ambiguity | keep occurrence patterns consistent |

Inference observes samples. Stable business contracts should be reviewed and enforced downstream.


# Common JSON failures

| Symptom | Likely reason | Correction |
|---|---|---|
| Table has wrapper fields, not orders | wrong JSONPath | use `$.orders[*]` |
| No schema / unknown format | malformed JSON or wrong path | validate the document and path |
| One giant record | array elements not selected | add `[*]` at the repeating array |
| Inconsistent field types | mixed values across records | normalize source or enforce schema later |
| Missing field in some rows | optional property | keep nullable or populate explicitly |
| Duplicate/extra tables | unrelated files in target | isolate prefix and add exclusions |
| JSON Lines fails with wrapped-array path | structure mismatch | use a classifier/path suited to actual layout |


# Common XML failures

| Symptom | Likely reason | Correction |
|---|---|---|
| No records | wrong row tag or case | use exact `invoice` tag |
| Entire document becomes one record | wrapper selected as row | select the repeating element |
| Parse error | unclosed tag or unescaped `&` | validate well-formed XML |
| Fields appear unexpectedly named | attribute/element representation | inspect schema and normalize later |
| One item becomes struct, many become array | inconsistent occurrence patterns | keep representative repeated elements |
| Outer attributes missing | row is nested below wrapper | carry them into each row or enrich later |
| Empty self-closing row fails | unsupported row shape | use explicit opening and closing row tags |


# Safe schema-change exercise

Use disposable copies in separate test prefixes:

1. Record current Catalog schemas.
2. Add `paymentMethod` to every JSON order.
3. Add `<paymentTerms>NET_15</paymentTerms>` to every XML invoice.
4. Run with normal update behavior and inspect additions.
5. Change the crawler to **add new columns only** and test another compatible field.
6. Change to **log/ignore** and confirm later differences do not overwrite the table.
7. Restore clean input and retain the intended table version.

Updating a classifier does not guarantee retroactive reclassification of objects already remembered by a crawler; a new crawler is safer for classifier corrections.


# Validation checklist

- [ ] Files are valid UTF-8 JSON/XML with correct extensions
- [ ] Each format has an isolated S3 prefix
- [ ] JSONPath selects `$.orders[*]`
- [ ] XML row tag is exactly `invoice`
- [ ] Correct classifier is attached first to each crawler
- [ ] Correct Region, database, IAM role, and S3 target
- [ ] Table location and classification are correct
- [ ] Nested objects, arrays, elements, and attributes are visible
- [ ] Money, dates, optional fields, and identifiers have suitable types
- [ ] Crawler logs contain no parsing or permission errors
- [ ] A downstream sample query confirms actual row interpretation


# Summary

```text
Staged document
   ↓
Record boundary
   ├── JSON: $.orders[*]
   └── XML: row tag = invoice
   ↓
Custom classifier
   ↓
Dataset-specific crawler
   ↓
Glue Catalog table with nested schema
   ↓
Validation before transformation
```

The critical decision is the **record boundary**. Once that boundary is correct, the crawler can describe nested content; later processing can flatten, type, validate, join, and curate it.

## References

- [AWS Glue: defining and managing classifiers](https://docs.aws.amazon.com/glue/latest/dg/add-classifier.html)
- [AWS Glue: custom JSON and XML classifiers](https://docs.aws.amazon.com/glue/latest/dg/custom-classifier.html)
- [AWS Glue: using crawlers to populate the Data Catalog](https://docs.aws.amazon.com/glue/latest/dg/add-crawler.html)
